In [1]:
import pandas as pd

In [2]:
combined_dataset = []

rpi3 = pd.read_csv('liboqs-scripts/signing_benchmark_rpi_3.csv')
rpi4 = pd.read_csv('liboqs-scripts/signing_benchmark_rpi_4.csv')
rpi5 = pd.read_csv('liboqs-scripts/signing_benchmark_rpi_5.csv')

energy_data = pd.read_csv('energy_data.csv')

In [3]:
rpi3['Start Time'] = pd.to_datetime(rpi3['Start Time'])
rpi3['End Time'] = pd.to_datetime(rpi3['End Time'])
rpi4['Start Time'] = pd.to_datetime(rpi4['Start Time'])
rpi4['End Time'] = pd.to_datetime(rpi4['End Time'])
rpi5['Start Time'] = pd.to_datetime(rpi5['Start Time'])
rpi5['End Time'] = pd.to_datetime(rpi5['End Time'])
energy_data['Timestamp'] = pd.to_datetime(energy_data['Timestamp'])

In [4]:
def combine_data(algorithm_data, energy_data, socket):
    
    for _, row in algorithm_data.iterrows():
        start_time = row['Start Time']
        end_time = row['End Time']
        avg_power = avg_current = avg_voltage = 0

        mask = (energy_data['Timestamp'] >= start_time) & (energy_data['Timestamp'] < end_time) & (energy_data['Socket'] == socket)
        filtered = energy_data[mask]

        if len(filtered) > 0:
            avg_power = filtered['Power (W)'].mean()
            avg_current = filtered['Current (A)'].mean()
            avg_voltage = filtered['Voltage (V)'].mean()

        combined_dataset.append({
            'Socket': socket,
            'Algorithm': row['Algorithm'],
            "Start Time": start_time,
            "End Time": end_time,
            "Execution Time": row['Execution Time (s)'],
            "Power": avg_power,
            "Current": avg_current,
            "Voltage": avg_voltage,
            "Memory": row['Memory Used (MB)'],
            "CPU Usage": row['CPU Usage (%)'],
            "Read Bytes": row['Read Bytes'],
            "Write Bytes": row['Write Bytes'],
            "Signature": row['Total Signature Size (bytes)'],
            "Num Files": row['Num Files'],
            "Size": row['Total Size MB'],
            "Avg File Size": row['Avg File Size MB'],
            "CPU Count": row['CPU Count'],
            "Total RAM": row['Total RAM MB'],
            "Available RAM": row['Available RAM MB'],
            "System CPU": row['System CPU %'],
            "System Memory": row['System Memory %'],
            "Time Per File": row['Avg Time Per File'],
            "Throughput": row['Throughput MB/s']
        })

    return pd.DataFrame(combined_dataset)

In [5]:
rpi3_df = combine_data(rpi3, energy_data, "rpi3")
rpi4_df = combine_data(rpi4, energy_data, "rpi4")
rpi5_df = combine_data(rpi5, energy_data, "rpi5")
all_datasets = pd.concat([rpi3_df, rpi4_df, rpi5_df], ignore_index=True)
all_datasets.to_csv('combined_dataset.csv', index=False)